# Stage 1 — NESO historic demand

This notebook is the Stage 1 demo surface. It imports the `bristol_ml.ingestion.neso` module, reuses a local cache if present (fetching from NESO's CKAN API otherwise), and plots two views of GB electricity demand.

The data is half-hourly at storage; hourly aggregation is a features-layer derivation, demonstrated here for display only (`.resample("h").mean()`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # cache_dir in conf/ingestion/neso.yaml resolves against cwd

import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

from bristol_ml import CachePolicy, load_config  # noqa: E402
from bristol_ml.ingestion import neso  # noqa: E402

cfg = load_config(config_path=REPO_ROOT / "conf")
assert cfg.ingestion.neso is not None, "NESO ingestion config not resolved"
path = neso.fetch(cfg.ingestion.neso, cache=CachePolicy.AUTO)
print(f"Cache path: {path}")

In [ ]:
df = neso.load(Path(path))
df = df.set_index("timestamp_utc").sort_index()
print(df.info())
df.head()

## A week of hourly demand

The canonical twin-peak shape — morning ramp, working-day plateau, evening peak — is the signal every subsequent model is trying to capture. Weekends flatten out.

In [ ]:
first_monday = df.index.to_series().loc[df.index.weekday == 0].iloc[0].normalize()
week_start = first_monday
week_end = week_start + pd.Timedelta(days=7)
hourly = df.loc[week_start:week_end, "nd_mw"].resample("h").mean()

fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(ax=ax, color="C0")
ax.set_ylabel("ND (MW)")
ax.set_xlabel("UTC")
ax.set_title(f"Hourly GB National Demand — week of {week_start.date()}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Daily peaks across the cached window

Daily maxima across whatever is in the cache. With the full configured catalogue (2018–2025) this is a multi-year view dominated by winter peaks and summer troughs; with a narrow cassette-seeded cache it is just the slice that happens to be present. The gentle long-term decline (when the full window is present) reflects improved energy efficiency and the growing contribution of distributed solar that is invisible to transmission metering — see the caveat below.

In [ ]:
daily_peak = df["nd_mw"].resample("D").max()

fig, ax = plt.subplots(figsize=(10, 4))
daily_peak.plot(ax=ax, color="C3")
ax.set_ylabel("Daily peak ND (MW)")
ax.set_xlabel("UTC")
ax.set_title("Daily peak GB National Demand")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Caveat — embedded generation

`ND` (National Demand) is what the transmission system sees. It excludes rooftop and farm-scale solar, small wind, and other *embedded* generation that sits on the distribution network. NESO estimates embedded wind and solar separately (`EMBEDDED_WIND_GENERATION`, `EMBEDDED_SOLAR_GENERATION`); those columns are deliberately not persisted here (Stage 1 intent §"Out of scope").

As distributed solar has grown, `ND` has drifted away from true GB electricity consumption — especially mid-day in summer. A rising baseline of invisible self-consumption sits under the curve above. If you add `EMBEDDED_SOLAR_GENERATION` back to `ND`, you'll recover a fuller picture of underlying demand. That trade-off is discussed in the research note at `docs/lld/research/01-neso-ingestion.md`.